## Step 1: Test API connection
Confirms the API key and base competition endpoint work before pulling real data.

In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FOOTBALL_DATA_API_KEY")

BASE_URL = "https://api.football-data.org/v4"
COMPETITION_CODE = "PL"
headers = {"X-Auth-Token": api_key}

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/features", exist_ok=True)

response = requests.get(f"{BASE_URL}/competitions/{COMPETITION_CODE}", headers=headers)
print(response.status_code)

200


## Step 2: Pull match data across multiple seasons
Rate-limited with a 6.5s delay between requests to stay within the free-tier API limits. Seasons are identified by their start year (2023 = the 2023/24 season).

In [2]:
seasons_to_pull = [2023, 2024, 2025]

all_seasons_matches = []
for year in seasons_to_pull:
    resp = requests.get(
        f"{BASE_URL}/competitions/{COMPETITION_CODE}/matches",
        headers=headers,
        params={"season": year}
    )
    data = resp.json()
    print(year, resp.status_code, len(data.get("matches", [])))
    all_seasons_matches.extend(data.get("matches", []))
    time.sleep(6.5)

2023 200 380
2024 200 380
2025 200 380


## Step 3: Pull team data across multiple seasons

In [3]:
all_teams = []
for year in seasons_to_pull:
    resp = requests.get(
        f"{BASE_URL}/competitions/{COMPETITION_CODE}/teams",
        headers=headers,
        params={"season": year}
    )
    all_teams.extend(resp.json()["teams"])
    time.sleep(6.5)

teams_df_multi = pd.DataFrame(all_teams).drop_duplicates(subset="id")
print(teams_df_multi.shape)

(25, 16)


## Step 4: Extract a clean season year before saving
The API returns `season` as a nested object (`{"startDate": "2023-08-11", ...}`) — extract just the year here, *before* saving, so every downstream notebook works with a clean value instead of a raw dict-string.

In [4]:
matches_df = pd.DataFrame(all_seasons_matches)
matches_df["season"] = matches_df["season"].apply(lambda x: x["startDate"][:4] if isinstance(x, dict) else x)
matches_df["season"].value_counts()

season
2023    380
2024    380
2025    380
Name: count, dtype: int64

## Step 5: Save multi-season raw datasets

In [5]:
teams_df_multi.to_csv("../data/raw/raw_teams_multi_season.csv", index=False)
matches_df.to_csv("../data/raw/raw_matches_multi_season.csv", index=False)
print(matches_df.shape)

(1140, 15)
